In [1]:
import pandas as pd
import os
import numpy as np
import re
import geopandas as gpd
from pathlib import Path

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\CB')

# Generate dummy IDs for records that are missing IDs

In [2]:
file_path = "original.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def infer_format(series):
    non_null = series.dropna().astype(str)
    if non_null.empty:
        return ("ID", 4)
    sample = non_null.iloc[0]
    m = re.match(r"^([A-Za-z]*)(\d+)$", sample)
    if m:
        prefix = m.group(1)
        width = len(m.group(2))
    else:
        prefix = ""
        width = max(len(re.findall(r"\d+", s)[-1]) if re.findall(r"\d+", s) else 4 for s in non_null)
    return (prefix, width)

def next_available(existing_nums, start=1):
    n = start
    while n in existing_nums:
        n += 1
    return n

def fill_ids(df, id_col, base_col):
    if id_col not in df.columns or base_col not in df.columns:
        return df

    # Ensure ID column can hold string values without dtype errors
    df[id_col] = df[id_col].astype("string")

    prefix, width = infer_format(df[id_col])
    existing = df[id_col].dropna().astype(str)

    def extract_num(x):
        m = re.match(r"^[A-Za-z]*?(\d+)$", x)
        return int(m.group(1)) if m else None

    used_nums = {n for n in (extract_num(x) for x in existing) if n is not None}

    # map base value -> existing id
    base_to_id = (
        df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
        .astype(str)
        .to_dict()
    )

    missing_mask = df[id_col].isna()
    for idx, row in df[missing_mask].iterrows():
        base_val = row[base_col]
        if pd.isna(base_val):
            continue
        if base_val in base_to_id:
            df.at[idx, id_col] = base_to_id[base_val]
        else:
            new_num = next_available(used_nums)
            used_nums.add(new_num)
            new_id = f"{prefix}{str(new_num).zfill(width)}"
            df.at[idx, id_col] = new_id
            base_to_id[base_val] = new_id

    # Remove trailing .0 from IDs that were read as floats
    df[id_col] = df[id_col].astype("string").str.replace(r"\.0$", "", regex=True)
    return df

biota = fill_ids(biota, "StationID", "Station")
biota = fill_ids(biota, "DeviceID", "Device")
biota = fill_ids(biota, "SampleID", "Sample")
biota = fill_ids(biota, "SubSampleID", "SubSample")
biota = fill_ids(biota, "SubsetID", "Subset")
biota = fill_ids(biota, "BiotaID", "Biota")

sheets["Biota"] = biota

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

In [3]:
# reload workbook
sheets = pd.read_excel(file_path, sheet_name=None)
biota = sheets.get("Biota")

def fill_ids_from_biota(df, biota_df, id_col, base_col):
    if df is None or biota_df is None:
        return df
    if id_col not in df.columns or base_col not in df.columns:
        return df

    mapping = (
        biota_df[[base_col, id_col]]
        .dropna()
        .drop_duplicates(subset=[base_col])
        .set_index(base_col)[id_col]
    )
    df[id_col] = df[id_col].fillna(df[base_col].map(mapping))
    return df

sheet_id_map = {
    "Station": [("StationID", "Station")],
    "Device": [("StationID", "Station"), ("DeviceID", "Device")],
    "Sample": [("StationID", "Station"), ("DeviceID", "Device"), ("SampleID", "Sample")],
    "Subsample": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
    ],
    "Subset": [
        ("StationID", "Station"),
        ("DeviceID", "Device"),
        ("SampleID", "Sample"),
        ("SubSampleID", "SubSample"),
        ("SubsetID", "Subset"),
    ],
}

for sheet_name, mappings in sheet_id_map.items():
    if sheet_name in sheets:
        df_sheet = sheets[sheet_name]
        for id_col, base_col in mappings:
            df_sheet = fill_ids_from_biota(df_sheet, biota, id_col, base_col)
        sheets[sheet_name] = df_sheet

with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

# Create long format

In [4]:
biota = pd.read_excel(file_path, sheet_name="Biota")
biota.to_parquet("cb_long.parquet", index=False)

# Key in necessary columns

Add Project column

In [5]:
biota = pd.read_parquet("cb_long.parquet")
subset_df = sheets.get("Subset", pd.DataFrame())

if "SubsetID" in biota.columns and "SubsetID" in subset_df.columns and "Project" in subset_df.columns:
    subset_proj = subset_df[["SubsetID", "Project"]].drop_duplicates(subset=["SubsetID"])
    biota = biota.merge(subset_proj, on="SubsetID", how="left", suffixes=("", "_subset"))
    if "Project_subset" in biota.columns:
        biota["Project"] = biota["Project"].fillna(biota["Project_subset"])
        biota = biota.drop(columns=["Project_subset"])
    biota.to_parquet("cb_long.parquet", index=False)

Add Title and Realm columns

In [6]:
biota = pd.read_parquet("cb_long.parquet")
dataset_df = sheets.get("Dataset", pd.DataFrame())
subset_df = sheets.get("Subset", pd.DataFrame())

if not dataset_df.empty and "Name" in dataset_df.columns and "Title" in dataset_df.columns and "Realm" in dataset_df.columns:
    project_to_meta = (
        dataset_df[["Name", "Title", "Realm"]]
        .dropna(subset=["Name"])
        .drop_duplicates(subset=["Name"])
        .set_index("Name")
    )

    subset_meta = subset_df.copy()
    subset_meta["Title"] = subset_meta["Project"].map(project_to_meta["Title"])
    subset_meta["Realm"] = subset_meta["Project"].map(project_to_meta["Realm"])

    subset_meta = subset_meta[["SubsetID", "Title", "Realm"]].drop_duplicates(subset=["SubsetID"])

    biota = biota.merge(subset_meta, on="SubsetID", how="left", suffixes=("", "_subset"))
    if "Title_subset" in biota.columns:
        biota["Title"] = biota["Title"].fillna(biota["Title_subset"])
        biota = biota.drop(columns=["Title_subset"])
    if "Realm_subset" in biota.columns:
        biota["Realm"] = biota["Realm"].fillna(biota["Realm_subset"])
        biota = biota.drop(columns=["Realm_subset"])

    biota.to_parquet("cb_long.parquet", index=False)

Add coordinates, depth, dates

In [7]:
biota = pd.read_parquet("cb_long.parquet")
station_df = pd.read_excel(file_path, sheet_name="Station")

cols = [
    "StationID",
    "LatStart", "LonStart", "LatEnd", "LonEnd",
    "DepthStart", "DepthEnd",
    "DateStart", "DateEnd",
]
station_coords = station_df[[c for c in cols if c in station_df.columns]].drop_duplicates(subset=["StationID"])

# Drop columns from biota that will come from station_coords (except StationID)
cols_to_drop = [c for c in cols[1:] if c in biota.columns and c in station_coords.columns]
if cols_to_drop:
    biota = biota.drop(columns=cols_to_drop)

biota = biota.merge(station_coords, on="StationID", how="left")

def preserve_date_value(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).isoformat()
    return str(value)

for col in ["DateStart", "DateEnd"]:
    if col in biota.columns:
        biota[col] = biota[col].map(preserve_date_value).astype("string")

biota.to_parquet("cb_long.parquet", index=False)


In [8]:
cb = pd.read_parquet("cb_long.parquet")

# Extract year from DateStart column using a few common date shapes
if "DateStart" in cb.columns:
    date_text = cb["DateStart"].astype("string").str.strip()
    cb["year"] = pd.Series(pd.NA, index=cb.index, dtype="Int64")

    # YYYY
    mask_year_only = date_text.str.fullmatch(r"\d{4}", na=False)
    cb.loc[mask_year_only, "year"] = date_text.loc[mask_year_only].astype("Int64")

    # DD.MM.YYYY
    remaining = cb["year"].isna() & date_text.notna() & (date_text != "")
    parsed = pd.to_datetime(date_text.loc[remaining], format="%d.%m.%Y", errors="coerce")
    cb.loc[remaining, "year"] = parsed.dt.year

    # Fallback for YYYY-MM-DD, timestamps, and other parseable variants
    remaining = cb["year"].isna() & date_text.notna() & (date_text != "")
    parsed = pd.to_datetime(date_text.loc[remaining], errors="coerce")
    cb.loc[remaining, "year"] = parsed.dt.year

    # Final fallback: pull the first 4-digit year out of any still-unparsed string
    remaining = cb["year"].isna() & date_text.notna() & (date_text != "")
    extracted_year = date_text.loc[remaining].str.extract(r"(\d{4})", expand=False)
    cb.loc[remaining, "year"] = pd.to_numeric(extracted_year, errors="coerce").astype("Int64")

    year_coverage = (cb["year"].notna().sum() / len(cb) * 100).round(2)
    print(f"Year coverage after extraction: {year_coverage:.2f}%")

    # Diagnostic: show the most common raw DateStart values that still could not be parsed
    unresolved = cb.loc[cb["year"].isna() & date_text.notna() & (date_text != ""), "DateStart"]
    unresolved_counts = unresolved.value_counts(dropna=False).head(20)
    print(f"Unresolved DateStart values: {len(unresolved):,}")
    if not unresolved_counts.empty:
        print("Top unresolved DateStart values:")
        print(unresolved_counts.to_string())
else:
    print("DateStart column not found; year column was not created")

cb.to_parquet("cb_long.parquet", index=False)
print("Added year column to cb_long.parquet")

Year coverage after extraction: 100.00%
Unresolved DateStart values: 0
Added year column to cb_long.parquet


Sample Remark

In [9]:
cb = pd.read_parquet("cb_long.parquet")
sample_df = pd.read_excel(file_path, sheet_name="Sample")

if "SampleID" in cb.columns and "SampleID" in sample_df.columns and "Remark" in sample_df.columns:
    sample_remark = sample_df[["SampleID", "Remark"]].drop_duplicates(subset=["SampleID"])
    sample_remark = sample_remark.rename(columns={"Remark": "SampleRemark"})
    cb = cb.merge(sample_remark, on="SampleID", how="left")

cb.to_parquet("cb_long.parquet", index=False)

Add taxonomic details

In [10]:
biota = pd.read_parquet("cb_long.parquet")
taxon_df = sheets.get("Taxon", pd.DataFrame())
biota_key = "AphiaId (Accepted)"
taxon_key = "ValidAphiaId"
cols_to_add = [
    taxon_key,
    "Status",
    "UnacceptReason",
    "Rank",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "ValidAuthority",
]

if not taxon_df.empty and taxon_key in taxon_df.columns and biota_key in biota.columns:
    taxon_meta = (
        taxon_df[[c for c in cols_to_add if c in taxon_df.columns]]
        .dropna(subset=[taxon_key])
        .drop_duplicates(subset=[taxon_key])
    )
    biota = biota.merge(
        taxon_meta,
        left_on=biota_key,
        right_on=taxon_key,
        how="left",
        suffixes=("", "_taxon"),
    )
    for col in cols_to_add[1:]:
        taxon_col = f"{col}_taxon"
        if taxon_col in biota.columns:
            if col in biota.columns:
                biota[col] = biota[col].fillna(biota[taxon_col])
            else:
                biota[col] = biota[taxon_col]
            biota = biota.drop(columns=[taxon_col])
    if taxon_key not in (biota_key,) and taxon_key in biota.columns:
        biota = biota.drop(columns=[taxon_key])

biota.to_parquet("cb_long.parquet", index=False)

Gear

In [11]:
biota = pd.read_parquet("cb_long.parquet")
device_df = sheets.get("Device", pd.DataFrame())

if "DeviceID" in biota.columns and "DeviceID" in device_df.columns and "Gear" in device_df.columns:
    device_gear = device_df[["DeviceID", "Gear"]].drop_duplicates(subset=["DeviceID"])
    biota = biota.merge(device_gear, on="DeviceID", how="left", suffixes=("", "_device"))
    if "Gear_device" in biota.columns:
        biota["Gear"] = biota["Gear"].fillna(biota["Gear_device"]) if "Gear" in biota.columns else biota["Gear_device"]
        biota = biota.drop(columns=["Gear_device"])
    biota.to_parquet("cb_long.parquet", index=False)

In [12]:
biota = pd.read_parquet("cb_long.parquet")
gear_df = sheets.get("Gear", pd.DataFrame())
device_df = sheets.get("Device", pd.DataFrame())

if not gear_df.empty and "Name" in gear_df.columns and "Gear" in device_df.columns:
    gear_cols = [c for c in ["Name", "Category", "Description", "MeshSize"] if c in gear_df.columns]
    gear_lookup = gear_df[gear_cols].drop_duplicates(subset=["Name"])

    device_gear_meta = (
        device_df[["DeviceID", "Gear"]]
        .dropna(subset=["DeviceID", "Gear"])
        .merge(gear_lookup, left_on="Gear", right_on="Name", how="left")
        .drop(columns=["Gear"])
        .drop_duplicates(subset=["DeviceID"])
    )

    biota = biota.merge(device_gear_meta, on="DeviceID", how="left", suffixes=("", "_gear"))
    for col in gear_cols:
        if col in biota.columns and f"{col}_gear" in biota.columns:
            biota[col] = biota[col].fillna(biota[f"{col}_gear"])
            biota = biota.drop(columns=[f"{col}_gear"])
        elif f"{col}_gear" in biota.columns:
            biota[col] = biota[f"{col}_gear"]
            biota = biota.drop(columns=[f"{col}_gear"])

    biota.to_parquet("cb_long.parquet", index=False)

individualCount

In [13]:
biota = pd.read_parquet("cb_long.parquet")

value_cols = [c for c in ["Count", "Abundance"] if c in biota.columns]
if value_cols and "BiotaID" in biota.columns:
    biota["individualCount"] = biota[value_cols].bfill(axis=1).iloc[:, 0]
    biota.to_parquet("cb_long.parquet", index=False)

areaSampled

In [14]:
biota = pd.read_parquet("cb_long.parquet")
sample_df = sheets.get("Sample", pd.DataFrame())
# Build areaSampled from Sample sheet values keyed by SampleID
if "SampleID" in biota.columns and "SampleID" in sample_df.columns:
    sample_effort = sample_df[["SampleID", "SampledArea", "ReferenceArea"]].drop_duplicates(subset=["SampleID"])
    biota = biota.merge(sample_effort, on="SampleID", how="left", suffixes=("", "_sample"))

    def format_effort(row):
        parts = []
        if pd.notna(row.get("ReferenceArea")):
            parts.append(row["ReferenceArea"])
        if pd.notna(row.get("SampledArea")):
            parts.append(row["SampledArea"])
        parts = [str(p) for p in parts]
        return "; ".join(parts) if parts else pd.NA

    biota["areaSampled"] = biota.apply(format_effort, axis=1).astype("string")
    biota = biota.drop(columns=[c for c in ["SampledArea", "ReferenceArea"] if c in biota.columns])

    biota.to_parquet("cb_long.parquet", index=False)

# Cut according to Arctic LMEs

In [15]:


cb = pd.read_parquet("cb_long.parquet")

# Load LME boundaries
lme_path = r"..\\LME_boundaries_2014\\LME_2013_polygon_projected.shp"
lme = gpd.read_file(lme_path).to_crs("EPSG:4326")

# Build points from coordinates and filter to within LME polygons
cb_gdf = gpd.GeoDataFrame(
    cb,
    geometry=gpd.points_from_xy(cb["LonStart"], cb["LatStart"]),
    crs="EPSG:4326",
)

lme_union = lme.union_all()
cb_in_lme = cb_gdf[cb_gdf.geometry.within(lme_union)].drop(columns=["geometry"])

cb_in_lme.to_parquet("cb_long.parquet", index=False)
print(f"Filtered cb_long.parquet to {len(cb_in_lme):,} rows within LME boundaries.")

Filtered cb_long.parquet to 243,609 rows within LME boundaries.


# Fix dates

In [16]:
cb = pd.read_parquet("cb_long.parquet")

# Extract day of year from DateStart and DateEnd using mixed format parsing
if "DateStart" in cb.columns:
    # Compare default parsing vs mixed-format parsing
    parsed_default_start = pd.to_datetime(cb["DateStart"], errors="coerce")
    parsed_mixed_start = pd.to_datetime(cb["DateStart"], errors="coerce", format="mixed")
    
    # Use mixed parser results to fill missing values from default parser
    cb["startDayOfYear"] = parsed_mixed_start.dt.dayofyear

if "DateEnd" in cb.columns:
    # Compare default parsing vs mixed-format parsing
    parsed_default_end = pd.to_datetime(cb["DateEnd"], errors="coerce")
    parsed_mixed_end = pd.to_datetime(cb["DateEnd"], errors="coerce", format="mixed")
    
    # Use mixed parser results to fill missing values from default parser
    cb["endDayOfYear"] = parsed_mixed_end.dt.dayofyear

cb.to_parquet("cb_long.parquet", index=False)
print("Added startDayOfYear and endDayOfYear columns using mixed datetime parsing and saved cb_long.parquet")

Added startDayOfYear and endDayOfYear columns using mixed datetime parsing and saved cb_long.parquet


# Map to final file

In [17]:
cb = pd.read_parquet("cb_long.parquet")

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec["column"], dtype_spec["final"]))

column_map = {
    "sourceID": "BiotaID",
    "datasetKey": "Project",
    "datasetTitle": "Title",
    "decimalLatitudeStart": "LatStart",
    "decimalLongitudeStart": "LonStart",
    "decimalLatitudeStop": "LatEnd",
    "decimalLongitudeStop": "LonEnd",
    "depthA_m": "DepthStart",
    "depthB_m": "DepthEnd",
    "eventID": "SubsetID",
    "waterBody": "Realm",
    "verbatimDate": "DateStart",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "year",
    "eventRemarks": "SampleRemark",
    "verbatimScientificName": "ScientificName (User)",
    "acceptedScientificName": "ScientificName (Accepted)",
    "identificationQualifier": "Specification",
    "taxonomicStatus": "Status",
    "taxonomicIssue": "UnacceptReason",
    "taxonRank": "Rank",
    "phylum": "Phylum",
    "class": "Class",
    "order": "Order",
    "family": "Family",
    "genus": "Genus",
    "validAuthority": "ValidAuthority",
    "aphiaID": "AphiaId (Accepted)",
    "individualCount": "individualCount",
    "areaSampled": "areaSampled",
    "biomass": "SumWet",
    "occurrenceRemarks": "Remark",
    "gear": "Gear",
    "gearCategory": "Category",
    "meshSize_mm": "MeshSize",
    "sieveFraction_µm": "SieveFraction",
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "volumeSampled_m3",
    "samplingEffort",
    "trawlingDistance_m",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment", 
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        return series.astype("boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


cb_final = pd.DataFrame(index=cb.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in cb.columns:
            cb_final[col] = coerce_series(cb[src], target_dtype)
        else:
            cb_final[col] = empty_series(cb.index, target_dtype)
            missing_sources.append(src)
    else:
        cb_final[col] = empty_series(cb.index, target_dtype)

cb_final = cb_final[output_columns]
cb_final.to_parquet("cb_final.parquet", index=False)

print(f"Saved cb_final.parquet with {len(cb_final):,} rows and {len(cb_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in cb:", sorted(set(missing_sources)))

Saved cb_final.parquet with 243,609 rows and 63 columns.


In [18]:
cb_final = pd.read_parquet("cb_final.parquet")
cb_final["sourceDatabase"] = "CRITTERBASE"
cb_final.to_parquet("cb_final.parquet", index=False)
print("Added sourceDatabase column with value 'CRITTERBASE' and saved cb_final.parquet.")

Added sourceDatabase column with value 'CRITTERBASE' and saved cb_final.parquet.


In [19]:
cb_final = pd.read_parquet("cb_final.parquet")

cb_final.loc[cb_final["depthA_m"].notna(), "depthAType"] = "start"
cb_final.loc[cb_final["depthB_m"].notna(), "depthBType"] = "end"
cb_final.loc[cb_final["areaSampled"].notna(), "areaSampledUnit"] = "m2"
cb_final.loc[cb_final["biomass"].notna(), "biomassUnit"] = "g/m2"

cb_final.to_parquet("cb_final.parquet", index=False)
cb_final[["depthA_m", "depthAType", "depthB_m", "depthBType", "individualCount", "areaSampledUnit", "biomass", "biomassUnit"]].head()

,depthA_m,depthAType,depthB_m,depthBType,individualCount,areaSampledUnit,biomass,biomassUnit
0,853.0,start,<NA>,<NA>,3.5,m2,<NA>,<NA>
1,1059.0,start,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,486.0,start,<NA>,<NA>,18.0,m2,<NA>,<NA>
3,486.0,start,<NA>,<NA>,28.0,m2,<NA>,<NA>
4,557.0,start,<NA>,<NA>,3.0,m2,<NA>,<NA>


# Check data types 

In [20]:
cb_final = pd.read_parquet("cb_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": cb_final.columns,
    "dtype": cb_final.dtypes.astype(str).values,
    "non_null_count": cb_final.notna().sum().values,
    "null_count": cb_final.isna().sum().values,
    "non_null_pct": (cb_final.notna().sum().values / len(cb_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("cb_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,243609,0,100.000000
44,aphiaID,string,243609,0,100.000000
4,datasetKey,string,243609,0,100.000000
5,datasetTitle,string,243609,0,100.000000
7,decimalLatitudeStart,Float64,243609,0,100.000000
8,decimalLongitudeStart,Float64,243609,0,100.000000
19,eventID,string,243609,0,100.000000
56,gear,string,243609,0,100.000000
57,gearCategory,string,243609,0,100.000000
3,sourceDatabase,str,243609,0,100.000000
